# Phase 1 — Consolidated Classification Notebook (TA-compliant)

All 4 models (ARX, BiGRU, BiRNN+Attention, BiRNN+Skip) on a single TA-compliant pipeline.

**Target:** next-day binary direction of wheat futures close (up/down).

**Data:**
- 30 most recent daily closing prices as lagged predictors (t-1 ... t-30).
- 31 FRED-MD macro variables (the exact TA list) with correct t-codes and 1-month publication lag, forward-filled to daily.
- No OHLCV indicators, no alternative data, no wavelet denoising. Those belong to Part 2 ablations.

**Validation:** 5-fold `TimeSeriesSplit`. Scaler fit per-fold on training portion only. No shuffling.

**Tuning:** Optuna (TPE + MedianPruner) on each deep model: hidden size, layers (1–2), dropout, lr, weight decay, batch size.

**GPU:** Colab A100. Each trial trains on `cuda:0`; Optuna orchestrator runs on CPU.

**Per-model metrics block (TA spec):** accuracy, per-class precision/recall, confusion matrix, F1, AUC, plus one visualization.

In [ ]:
!pip install -q optuna

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score, f1_score)
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import warnings; warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print('Device:', DEVICE)

## 1. Base class (required by VIP abstract-class spec)

In [ ]:
class BaseForecastModel(ABC):
    def __init__(self, task_type, **hyperparameters):
        self.task_type = task_type
        self.hyperparameters = hyperparameters
    @abstractmethod
    def fit(self, X_train, y_train): pass
    @abstractmethod
    def predict(self, X): pass
    @abstractmethod
    def evaluate(self, X_test, y_test): pass
    @abstractmethod
    def save(self, fp): pass
    @abstractmethod
    def load(self, fp): pass

## 2. Paths + Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Quants ')
WHEAT_18 = BASE / 'Investing.com' / 'US Wheat Futures Historical Data_2018.csv'
WHEAT_25 = BASE / 'Investing.com' / 'US Wheat Futures Historical Data_2025.csv'
FRED = BASE / 'FredMD_Dataset' / '2025-10-MD.csv'

## 3. Load wheat close price

In [ ]:
def load_close(path):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    df['Price'] = df['Price'].astype(str).str.replace(',', '', regex=False).astype(float)
    return df['Price']

price = (pd.concat([load_close(WHEAT_18), load_close(WHEAT_25)])
         .pipe(lambda s: s[~s.index.duplicated(keep='last')])
         .sort_index())
price.name = 'Close'
print('Close price series:', price.shape, price.index.min().date(), '..', price.index.max().date())

## 4. Load FRED-MD — exact 31 TA variables, apply t-codes, add 1-month publication lag, forward-fill daily

In [ ]:
FEATURES_31 = [
    "RPI", "W875RX1", "CMRMTSPLx", "IPFPNSS", "USWTRADE", "USTRADE",
    "BUSLOANS", "CONSPI", "S&P 500", "S&P PE ratio", "FEDFUNDS",
    "TB3MS", "TB6MS", "GS1", "GS5", "GS10", "AAA", "BAA",
    "TB3SMFFM", "TB6SMFFM", "T1YFFM", "T5YFFM", "T10YFFM",
    "AAAFFM", "BAAFFM", "EXSZUSx", "EXJPUSx", "EXUSUKx", "EXCAUSx",
    "PPICMM", "UMCSENTx",
]
assert len(FEATURES_31) == 31

def apply_tcode(series, t):
    s = series.astype(float)
    if t == 1:   return s
    if t == 2:   return s.diff()
    if t == 3:   return s.diff().diff()
    if t == 4:   return np.log(s.clip(lower=1e-10))
    if t == 5:   return np.log(s.clip(lower=1e-10)).diff()
    if t == 6:   return np.log(s.clip(lower=1e-10)).diff().diff()
    if t == 7:   return (s / s.shift(1) - 1).diff()
    return s

tcodes_row = pd.read_csv(FRED, nrows=1)
fred_raw = pd.read_csv(FRED, skiprows=[1])
fred_raw['sasdate'] = pd.to_datetime(fred_raw['sasdate'], format='%m/%d/%Y')
fred_raw = fred_raw.set_index('sasdate').sort_index()

missing = [c for c in FEATURES_31 if c not in fred_raw.columns]
assert not missing, f'Missing FRED columns: {missing}'

tcodes = {c: int(tcodes_row[c].iloc[0]) for c in FEATURES_31}
print('t-codes:', tcodes)

fred_trans = pd.DataFrame({c: apply_tcode(fred_raw[c], tcodes[c]) for c in FEATURES_31})
fred_trans = fred_trans.replace([np.inf, -np.inf], np.nan).dropna()

# TA rule: 1-month publication delay — month-M data is only available in month M+1.
# Shift the index by +1 month so that when we forward-fill to daily, each day
# carries the PREVIOUS month's macro values (e.g. all Feb days carry Jan data).
fred_trans.index = fred_trans.index + pd.DateOffset(months=1)

daily_index = pd.date_range(fred_trans.index.min(), price.index.max(), freq='D')
fred_daily = fred_trans.reindex(daily_index).ffill()
print('FRED daily (forward-filled):', fred_daily.shape)

## 5. Align price + macro, build target, build rolling-window tensor

At time t:
- Input sequence: 30 timesteps (t-30 .. t-1), each with [close_lag, 31 macro vars of that day's month].
- Target: `y_t = 1 if close_t > close_{t-1} else 0` (next-day direction).
- The close_{t-1} used for y is *revealed* at time t; features at t use only t-1 and earlier, so no leakage.

In [ ]:
df = pd.concat([price, fred_daily], axis=1, join='inner').dropna()
df = df.sort_index()
print('Aligned shape:', df.shape)

closes = df['Close'].values
macros = df[FEATURES_31].values.astype(np.float32)
dates = df.index

# Binary next-day direction: y[t] = 1 if close[t] > close[t-1] else 0
y_all = (np.diff(closes) > 0).astype(np.int64)  # length N-1, indexed at t from 1..N-1

# Build rolling 30-day windows. For prediction at index t (in original df),
# features = (close_{t-30}..close_{t-1}, macro_{t-30}..macro_{t-1}); target = y at t.
LOOKBACK = 30
X_list, y_list, idx_list = [], [], []
for t in range(LOOKBACK, len(df)):
    price_window = closes[t-LOOKBACK:t].reshape(-1, 1)         # (30, 1)
    macro_window = macros[t-LOOKBACK:t]                        # (30, 31)
    step = np.concatenate([price_window, macro_window], axis=1).astype(np.float32)  # (30, 32)
    X_list.append(step)
    y_list.append(1 if closes[t] > closes[t-1] else 0)
    idx_list.append(dates[t])

X_seq = np.stack(X_list)                # (N, 30, 32)
y     = np.array(y_list, dtype=np.int64)
idx   = pd.DatetimeIndex(idx_list)

# Flat view for ARX / logistic: (N, 30*32) = (N, 960)
X_flat = X_seq.reshape(X_seq.shape[0], -1)

print('X_seq:', X_seq.shape, '  X_flat:', X_flat.shape, '  y:', y.shape,
      '  class balance up:', y.mean().round(4))
print('Date range:', idx.min().date(), '..', idx.max().date())

## 6. 5-fold TimeSeriesSplit and per-fold scaling

Scaler is fit on TRAIN features of each fold only, then applied to the VAL fold. The 3D sequence is scaled by flattening the last two dims, scaling, and reshaping back.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(np.arange(len(y))))
for i, (tr, va) in enumerate(splits):
    print(f'Fold {i+1}: train n={len(tr)}  [{idx[tr[0]].date()}..{idx[tr[-1]].date()}]  '
          f'val n={len(va)}  [{idx[va[0]].date()}..{idx[va[-1]].date()}]  '
          f'val class balance up={y[va].mean():.3f}')

def scale_fold_seq(X_tr, X_va):
    # Fit on train only; last-axis (feature) z-score across flattened (N*T, F).
    N_tr, T, F = X_tr.shape
    sc = StandardScaler().fit(X_tr.reshape(-1, F))
    X_tr_s = sc.transform(X_tr.reshape(-1, F)).reshape(N_tr, T, F).astype(np.float32)
    X_va_s = sc.transform(X_va.reshape(-1, F)).reshape(X_va.shape[0], T, F).astype(np.float32)
    return X_tr_s, X_va_s

def scale_fold_flat(X_tr, X_va):
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr).astype(np.float32), sc.transform(X_va).astype(np.float32)

## 7. Shared PyTorch training loop

In [ ]:
def orth_init_rnn(rnn):
    for name, p in rnn.named_parameters():
        if 'weight_hh' in name:
            nn.init.orthogonal_(p)
        elif 'weight_ih' in name:
            nn.init.kaiming_normal_(p)
        elif 'bias' in name:
            nn.init.zeros_(p)

def train_torch_classifier(model, X_tr, y_tr, X_va, y_va,
                           epochs=60, batch_size=64, lr=1e-3, weight_decay=1e-4,
                           patience=8, clip=1.0, verbose=False):
    model = model.to(DEVICE)
    pw = torch.tensor([(1 - y_tr.mean()) / max(y_tr.mean(), 1e-6)], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr.astype(np.float32)))
    ld = DataLoader(ds, batch_size=batch_size, shuffle=True)
    X_va_t = torch.from_numpy(X_va).to(DEVICE)
    best_auc, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in ld:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(xb).squeeze(-1)
            loss = loss_fn(logit, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(X_va_t).squeeze(-1)).cpu().numpy()
        try:
            auc = roc_auc_score(y_va, p_va)
        except ValueError:
            auc = 0.5
        if auc > best_auc:
            best_auc, best_state, bad = auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        p_va = torch.sigmoid(model(X_va_t).squeeze(-1)).cpu().numpy()
    return model, p_va, best_auc

## 8. TA-required metrics block

In [ ]:
def report_metrics(name, y_true, y_prob, threshold=0.5, show_plot=True):
    y_pred = (y_prob > threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1s, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float('nan')
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f'\n=== {name} ===')
    print(f'Accuracy : {acc:.4f}')
    print(f'AUC      : {auc:.4f}')
    print(f'F1       : {f1:.4f}')
    print(f'Precision [down, up] : [{prec[0]:.4f}, {prec[1]:.4f}]')
    print(f'Recall    [down, up] : [{rec[0]:.4f}, {rec[1]:.4f}]')
    print(f'Confusion matrix:\n{cm}')
    if show_plot:
        fig, ax = plt.subplots(1, 1, figsize=(4, 3.5))
        im = ax.imshow(cm, cmap='Blues')
        ax.set_title(f'{name} — confusion matrix')
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['down', 'up']); ax.set_yticklabels(['down', 'up'])
        ax.set_xlabel('predicted'); ax.set_ylabel('true')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, cm[i, j], ha='center', va='center',
                        color='white' if cm[i, j] > cm.max()/2 else 'black')
        plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout(); plt.show()
    return {'name': name, 'acc': acc, 'auc': auc, 'f1': f1,
            'prec_dn': prec[0], 'prec_up': prec[1],
            'rec_dn': rec[0],   'rec_up': rec[1]}

## 9. Model 1 — ARX (logistic regression on flattened 30×32)

ARX: linear classifier on the flat 960-dim vector (30 lagged prices + 30-timestep macro panel).

In [ ]:
class ARXClassifier(BaseForecastModel):
    def __init__(self, C=1.0):
        super().__init__(task_type='classification', C=C)
        self.C = C
        self.clf = None
    def fit(self, X_train, y_train):
        self.clf = LogisticRegression(C=self.C, max_iter=3000, solver='liblinear')
        self.clf.fit(X_train, y_train)
        return self
    def predict(self, X):
        return self.clf.predict(X)
    def predict_proba(self, X):
        return self.clf.predict_proba(X)[:, 1]
    def evaluate(self, X_test, y_test):
        p = self.predict_proba(X_test)
        return {'accuracy': accuracy_score(y_test, p > 0.5),
                'auc': roc_auc_score(y_test, p),
                'f1':  f1_score(y_test, (p > 0.5).astype(int))}
    def save(self, fp):
        import pickle; pickle.dump({'C': self.C, 'clf': self.clf}, open(fp, 'wb'))
    def load(self, fp):
        import pickle; d = pickle.load(open(fp, 'rb')); self.C = d['C']; self.clf = d['clf']

In [ ]:
def cv_arx(C_grid=(0.01, 0.1, 1.0, 10.0)):
    # pick best C via mean val AUC across folds
    best_C, best_mean_auc = None, -1
    for C in C_grid:
        aucs = []
        for tr, va in splits:
            X_tr, X_va = scale_fold_flat(X_flat[tr], X_flat[va])
            m = ARXClassifier(C=C).fit(X_tr, y[tr])
            p_va = m.predict_proba(X_va)
            aucs.append(roc_auc_score(y[va], p_va))
        if np.mean(aucs) > best_mean_auc:
            best_mean_auc, best_C = float(np.mean(aucs)), C
    # final evaluation: refit per fold with best_C, aggregate val predictions across all folds
    all_y, all_p = [], []
    per_fold = []
    for tr, va in splits:
        X_tr, X_va = scale_fold_flat(X_flat[tr], X_flat[va])
        m = ARXClassifier(C=best_C).fit(X_tr, y[tr])
        p_va = m.predict_proba(X_va)
        per_fold.append(accuracy_score(y[va], p_va > 0.5))
        all_y.append(y[va]); all_p.append(p_va)
    all_y = np.concatenate(all_y); all_p = np.concatenate(all_p)
    print(f'[ARX] best C = {best_C}, per-fold acc = {[round(a,4) for a in per_fold]}, mean = {np.mean(per_fold):.4f}')
    return all_y, all_p, {'best_C': best_C, 'fold_accs': per_fold}

y_arx, p_arx, info_arx = cv_arx()
metrics_arx = report_metrics('ARX (logistic)', y_arx, p_arx)

## 10. BiGRU — **pooling bug fixed** (last hidden state)

In [ ]:
class BiGRUClassifier(BaseForecastModel, nn.Module):
    def __init__(self, n_features, hidden=32, layers=1, dropout=0.3):
        nn.Module.__init__(self)
        BaseForecastModel.__init__(self, task_type='classification',
                                   hidden=hidden, layers=layers, dropout=dropout)
        self.rnn = nn.GRU(n_features, hidden, num_layers=layers, batch_first=True,
                          bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        orth_init_rnn(self.rnn)
        self.head = nn.Sequential(nn.LayerNorm(2*hidden), nn.Dropout(dropout),
                                  nn.Linear(2*hidden, 1))
    def forward(self, x):
        out, _ = self.rnn(x)         # (B, T, 2h)
        last = out[:, -1, :]         # last hidden state (bug fix: NOT mean over time)
        return self.head(last)
    # BaseForecastModel stubs — fit/evaluate are driven by the CV loop below
    def fit(self, X_train, y_train): raise NotImplementedError('use cv_deep()')
    def predict(self, X):
        self.eval()
        with torch.no_grad():
            return (torch.sigmoid(self(torch.from_numpy(X).to(DEVICE)).squeeze(-1))
                    .cpu().numpy() > 0.5).astype(int)
    def evaluate(self, X_test, y_test):
        p = torch.sigmoid(self(torch.from_numpy(X_test).to(DEVICE)).squeeze(-1)).detach().cpu().numpy()
        return {'accuracy': accuracy_score(y_test, p > 0.5), 'auc': roc_auc_score(y_test, p)}
    def save(self, fp): torch.save(self.state_dict(), fp)
    def load(self, fp): self.load_state_dict(torch.load(fp, map_location=DEVICE))

## 11. BiRNN + Attention — **softmax over TIME dim** (bug fix)

In [ ]:
class BiRNNAttnClassifier(BaseForecastModel, nn.Module):
    def __init__(self, n_features, hidden=32, layers=1, dropout=0.3):
        nn.Module.__init__(self)
        BaseForecastModel.__init__(self, task_type='classification',
                                   hidden=hidden, layers=layers, dropout=dropout)
        self.rnn = nn.GRU(n_features, hidden, num_layers=layers, batch_first=True,
                          bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        orth_init_rnn(self.rnn)
        self.attn_score = nn.Linear(2*hidden, 1)
        self.head = nn.Sequential(nn.LayerNorm(2*hidden), nn.Dropout(dropout),
                                  nn.Linear(2*hidden, 1))
    def forward(self, x):
        out, _ = self.rnn(x)                              # (B, T, 2h)
        scores = self.attn_score(out)                     # (B, T, 1)
        weights = torch.softmax(scores, dim=1)            # softmax over TIME (bug fix)
        ctx = (weights * out).sum(dim=1)                  # (B, 2h)
        return self.head(ctx)
    def fit(self, X_train, y_train): raise NotImplementedError('use cv_deep()')
    def predict(self, X):
        self.eval()
        with torch.no_grad():
            return (torch.sigmoid(self(torch.from_numpy(X).to(DEVICE)).squeeze(-1))
                    .cpu().numpy() > 0.5).astype(int)
    def evaluate(self, X_test, y_test):
        p = torch.sigmoid(self(torch.from_numpy(X_test).to(DEVICE)).squeeze(-1)).detach().cpu().numpy()
        return {'accuracy': accuracy_score(y_test, p > 0.5), 'auc': roc_auc_score(y_test, p)}
    def save(self, fp): torch.save(self.state_dict(), fp)
    def load(self, fp): self.load_state_dict(torch.load(fp, map_location=DEVICE))

## 12. BiRNN + Skip — explicit skip projection to matching dim

In [ ]:
class BiRNNSkipClassifier(BaseForecastModel, nn.Module):
    def __init__(self, n_features, hidden=32, layers=1, dropout=0.3):
        nn.Module.__init__(self)
        BaseForecastModel.__init__(self, task_type='classification',
                                   hidden=hidden, layers=layers, dropout=dropout)
        self.rnn1 = nn.GRU(n_features, hidden, batch_first=True, bidirectional=True)
        self.skip_proj = nn.Linear(n_features, 2*hidden)   # match residual dim explicitly
        self.drop = nn.Dropout(dropout)
        self.use_second = (layers == 2)
        if self.use_second:
            self.rnn2 = nn.GRU(2*hidden, hidden, batch_first=True, bidirectional=True)
            orth_init_rnn(self.rnn2)
        orth_init_rnn(self.rnn1)
        self.head = nn.Sequential(nn.LayerNorm(2*hidden), nn.Dropout(dropout),
                                  nn.Linear(2*hidden, 1))
    def forward(self, x):
        r1, _ = self.rnn1(x)                    # (B, T, 2h)
        r1 = self.drop(r1 + self.skip_proj(x))  # residual from raw input, shapes explicit
        if self.use_second:
            r2, _ = self.rnn2(r1)
            r1 = r2 + r1                        # stage-2 residual
        return self.head(r1[:, -1, :])
    def fit(self, X_train, y_train): raise NotImplementedError('use cv_deep()')
    def predict(self, X):
        self.eval()
        with torch.no_grad():
            return (torch.sigmoid(self(torch.from_numpy(X).to(DEVICE)).squeeze(-1))
                    .cpu().numpy() > 0.5).astype(int)
    def evaluate(self, X_test, y_test):
        p = torch.sigmoid(self(torch.from_numpy(X_test).to(DEVICE)).squeeze(-1)).detach().cpu().numpy()
        return {'accuracy': accuracy_score(y_test, p > 0.5), 'auc': roc_auc_score(y_test, p)}
    def save(self, fp): torch.save(self.state_dict(), fp)
    def load(self, fp): self.load_state_dict(torch.load(fp, map_location=DEVICE))

## 13. Optuna CV for deep models — shared objective

In [ ]:
N_FEATURES = X_seq.shape[-1]   # 32

def suggest_hparams(trial):
    return {
        'hidden':       trial.suggest_categorical('hidden', [16, 24, 32, 48]),
        'layers':       trial.suggest_categorical('layers', [1, 2]),
        'dropout':      trial.suggest_float('dropout', 0.2, 0.6),
        'lr':           trial.suggest_float('lr', 1e-4, 3e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        'batch_size':   trial.suggest_categorical('batch_size', [32, 64, 128]),
    }

def cv_deep(model_cls, n_trials=30, name='model'):
    def objective(trial):
        hp = suggest_hparams(trial)
        fold_aucs = []
        for i, (tr, va) in enumerate(splits):
            X_tr, X_va = scale_fold_seq(X_seq[tr], X_seq[va])
            model = model_cls(n_features=N_FEATURES, hidden=hp['hidden'],
                              layers=hp['layers'], dropout=hp['dropout'])
            _, p_va, _ = train_torch_classifier(
                model, X_tr, y[tr], X_va, y[va],
                epochs=60, batch_size=hp['batch_size'], lr=hp['lr'],
                weight_decay=hp['weight_decay'], patience=8, clip=1.0)
            try:
                auc = roc_auc_score(y[va], p_va)
            except ValueError:
                auc = 0.5
            fold_aucs.append(auc)
            trial.report(auc, i)
            if trial.should_prune():
                raise optuna.TrialPruned()
            del model; torch.cuda.empty_cache()
        return float(np.mean(fold_aucs))

    study = optuna.create_study(direction='maximize',
                                sampler=TPESampler(seed=SEED),
                                pruner=MedianPruner(n_warmup_steps=2))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'[{name}] best val AUC = {study.best_value:.4f}')
    print(f'[{name}] best params  = {study.best_params}')

    # Aggregate out-of-fold predictions using best hparams
    hp = study.best_params
    all_y, all_p, per_fold = [], [], []
    for tr, va in splits:
        X_tr, X_va = scale_fold_seq(X_seq[tr], X_seq[va])
        model = model_cls(n_features=N_FEATURES, hidden=hp['hidden'],
                          layers=hp['layers'], dropout=hp['dropout'])
        _, p_va, _ = train_torch_classifier(
            model, X_tr, y[tr], X_va, y[va],
            epochs=60, batch_size=hp['batch_size'], lr=hp['lr'],
            weight_decay=hp['weight_decay'], patience=8, clip=1.0)
        per_fold.append(accuracy_score(y[va], p_va > 0.5))
        all_y.append(y[va]); all_p.append(p_va)
        del model; torch.cuda.empty_cache()
    print(f'[{name}] per-fold acc = {[round(a,4) for a in per_fold]}, mean = {np.mean(per_fold):.4f}')
    return np.concatenate(all_y), np.concatenate(all_p), {
        'best_auc': study.best_value, 'best_params': hp, 'fold_accs': per_fold}

## 14. Run Optuna — BiGRU

In [ ]:
y_bg, p_bg, info_bg = cv_deep(BiGRUClassifier, n_trials=30, name='BiGRU')
metrics_bg = report_metrics('BiGRU', y_bg, p_bg)

## 15. Run Optuna — BiRNN + Attention

In [ ]:
y_at, p_at, info_at = cv_deep(BiRNNAttnClassifier, n_trials=30, name='BiRNN+Attn')
metrics_at = report_metrics('BiRNN+Attention', y_at, p_at)

## 16. Run Optuna — BiRNN + Skip

In [ ]:
y_sk, p_sk, info_sk = cv_deep(BiRNNSkipClassifier, n_trials=30, name='BiRNN+Skip')
metrics_sk = report_metrics('BiRNN+Skip', y_sk, p_sk)

## 17. Summary table (all 4 models)

In [ ]:
summary = pd.DataFrame([metrics_arx, metrics_bg, metrics_at, metrics_sk])
summary = summary[['name', 'acc', 'auc', 'f1', 'prec_up', 'rec_up', 'prec_dn', 'rec_dn']]
print(summary.to_string(index=False))
summary.to_csv('/content/drive/MyDrive/Quants /phase1_summary.csv', index=False)
print('\nSaved: /content/drive/MyDrive/Quants /phase1_summary.csv')

# Bar plot of mean accuracy across models
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(summary['name'], summary['acc'], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
ax.axhline(0.5, linestyle='--', color='gray', label='coin toss')
ax.axhline(0.52, linestyle=':', color='black', label='target = 0.52')
ax.set_ylabel('Accuracy (5-fold mean)')
ax.set_ylim(0.40, max(0.60, summary['acc'].max() + 0.03))
ax.set_title('Test accuracy by model')
ax.legend()
for i, v in enumerate(summary['acc']):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.show()